# 08. External Validation — CheXpert (Domain Shift 분석)

## 목적
NIH ChestX-ray14로 학습한 모델을 **Stanford CheXpert** 데이터셋에 적용해  
도메인 시프트(Domain Shift) 크기를 정량화합니다.

## NIH ↔ CheXpert 공통 클래스 (7개)
| NIH 레이블 | CheXpert 컬럼 |
|-----------|---------------|
| Atelectasis | Atelectasis |
| Cardiomegaly | Cardiomegaly |
| Consolidation | Consolidation |
| Edema | Edema |
| Effusion | Pleural Effusion |
| Pneumonia | Pneumonia |
| Pneumothorax | Pneumothorax |

## Uncertain(-1) 처리
기본: **U-zeros** (−1 → 0, 보수적 평가)

In [1]:
# ── 1. 환경 및 경로 설정 ──────────────────────────────────────────────────
import os
import sys
from pathlib import Path

PROJECT_DIR = r"C:\CSE4187\cxr-cad"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f"현재 작업 폴더: {os.getcwd()}")

현재 작업 폴더: C:\CSE4187\cxr-cad


In [2]:
# ── 2. 라이브러리 임포트 및 변수 설정 ───────────────────────────────────────
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import yaml

# 커스텀 모듈 임포트
from src.preprocess.chexpert_loader import build_chexpert_val_loader, EVAL_LABELS
from src.preprocess.transforms import get_inference_transforms
from src.train.models import build_model, DISEASE_LABELS
from src.analysis.evaluation import compute_auroc, compute_auprc
from src.analysis.external_val import compare_internal_external

# config 로드
with open('configs/config.yaml') as f:
    CFG = yaml.safe_load(f)

# 평가할 모델 리스트
MODELS_TO_EVAL = ['densenet', 'efficientnet', 'vit']

# 로컬 CheXpert 경로 설정
CHEXPERT_ROOT = r"data\chexpert\CheXpert-v1.0-small"

# 기타 파라미터
IMAGE_SIZE = CFG['data']['image_size']
UNCERTAIN = CFG['chexpert']['uncertain_strategy']  
EVAL_SPLIT = CFG['chexpert']['eval_split']         
BATCH_SIZE = CFG['train']['batch_size']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"CheXpert 경로: {CHEXPERT_ROOT}")
print(f"평가 대상 모델: {MODELS_TO_EVAL}")

CheXpert 경로: data\chexpert\CheXpert-v1.0-small
평가 대상 모델: ['densenet', 'efficientnet', 'vit']


In [3]:
# ── 3. CheXpert 데이터 로드 ────────────────────────────────────────────────
chex_loader, eval_labels = build_chexpert_val_loader(
    chexpert_root=CHEXPERT_ROOT,
    split=EVAL_SPLIT,
    uncertain_strategy=UNCERTAIN,
    transform=get_inference_transforms(IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    num_workers=0, # 윈도우 환경 병렬 로딩 에러 방지
)

print(f"\n외부 검증 데이터 로드 완료. 평가 라벨 ({len(eval_labels)}개): {eval_labels}")

[CheXpert] valid.csv 로드 완료
  총 샘플: 202 (uncertain_strategy=u_zeros, frontal_only=True)

외부 검증 데이터 로드 완료. 평가 라벨 (7개): ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Pneumonia', 'Pneumothorax']


In [4]:
# ── 4. 18개 전체 가중치 파일 순차 검증 및 결과 자동 저장 ────────────────────────
eval_indices = [DISEASE_LABELS.index(l) for l in EVAL_LABELS]

for model_key in MODELS_TO_EVAL:
    CHECKPOINT_DIR = Path("checkpoints") / model_key
    
    # 해당 모델 폴더 내의 모든 .pth 파일을 자동으로 찾기 (알파벳 정렬)
    ckpt_files = sorted(list(CHECKPOINT_DIR.glob("*.pth")))
    
    if not ckpt_files:
        print(f"\n" + "="*65)
        print(f"[{model_key.upper()}] 폴더에 .pth 파일이 없어 건너뜁니다.")
        print("="*65)
        continue

    # 찾은 각각의 파일(예: densenet_best.pth, vit_fold1_best.pth 등)에 대해 반복
    for CKPT_PATH in ckpt_files:
        ckpt_name = CKPT_PATH.stem # 파일명 확장자 제외 추출
        
        print(f"\n" + "="*65)
        print(f"[{ckpt_name}] 도메인 시프트 검증 시작")
        print("="*65)
        
        # 내부 검증 결과 파일 (test_predictions.csv) 확인
        NIH_CSV_PATH = CHECKPOINT_DIR / "test_predictions.csv"
        if not NIH_CSV_PATH.exists():
            print(f"'{NIH_CSV_PATH}' 파일이 없어 내부(NIH) 성능과 비교할 수 없습니다. 건너뜁니다.")
            continue

        # 1. 모델 뼈대 생성 및 가중치 로드
        model = build_model(model_key)
        ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True)
        
        # 2. 멀티 GPU(DataParallel) 'module.' 에러 방지용 정제 로직
        state_dict = ckpt.get('model_state_dict', ckpt)
        clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
            
        model.load_state_dict(clean_state_dict)
        model.to(DEVICE)
        model.eval()

        # 3. CheXpert 데이터 추론
        all_probs, all_targets = [], []
        with torch.no_grad():
            for images, labels in tqdm(chex_loader, desc=f'{ckpt_name} 추론 중'):
                images = images.to(device=DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    logits = model(images)
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
                all_targets.append(labels.numpy())

        y_true_14 = np.concatenate(all_targets, axis=0)
        y_prob_14 = np.concatenate(all_probs, axis=0)

        # 4. CheXpert AUROC 계산
        y_true_7 = y_true_14[:, eval_indices]
        y_prob_7 = y_prob_14[:, eval_indices]
        chex_auroc = compute_auroc(y_true_7, y_prob_7, EVAL_LABELS)
        
        # 5. 내부 NIH AUROC 로드 및 계산
        nih_df = pd.read_csv(NIH_CSV_PATH)
        nih_true_7 = nih_df[[f"{l}_true" for l in EVAL_LABELS]].values
        nih_prob_7 = nih_df[[f"{l}_prob" for l in EVAL_LABELS]].values
        nih_auroc = compute_auroc(nih_true_7, nih_prob_7, EVAL_LABELS)
        
        # 6. 갭(Gap) 계산 및 표 생성
        gap = compare_internal_external(nih_auroc, chex_auroc, EVAL_LABELS)
        rows = []
        for label in EVAL_LABELS:
            rows.append({
                'Disease': label,
                'NIH AUROC': round(nih_auroc.get(label, float('nan')), 4),
                'CheXpert AUROC': round(chex_auroc.get(label, float('nan')), 4),
                'Gap': f"{gap.get(label, float('nan')):+.1%}"
            })
        rows.append({
            'Disease': 'macro_avg',
            'NIH AUROC': round(nih_auroc['macro_avg'], 4),
            'CheXpert AUROC': round(chex_auroc['macro_avg'], 4),
            'Gap': f"{gap['macro_avg']:+.1%}"
        })
        
        cmp_df = pd.DataFrame(rows)
        print(f"\n[ {ckpt_name} 분석 결과 ]")
        print(cmp_df.to_string(index=False))
        
        # 7. 결과 CSV 저장 (각 가중치 파일명으로 개별 저장)
        save_csv_path = CHECKPOINT_DIR / f"{ckpt_name}_domain_shift.csv"
        cmp_df.to_csv(save_csv_path, index=False)
        
        # 8. 차트 시각화
        nih_vals  = [nih_auroc[l]  for l in EVAL_LABELS]
        chex_vals = [chex_auroc[l] for l in EVAL_LABELS]
        x = range(len(EVAL_LABELS))
        width = 0.35
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        
        # 바 차트
        ax = axes[0]
        ax.bar([i - width/2 for i in x], nih_vals,  width, label='NIH (Internal)', color='steelblue')
        ax.bar([i + width/2 for i in x], chex_vals, width, label='CheXpert (External)', color='coral')
        ax.set_xticks(list(x))
        ax.set_xticklabels(EVAL_LABELS, rotation=40, ha='right', fontsize=9)
        ax.set_ylim(0.5, 1.0)
        ax.set_ylabel('AUROC')
        ax.set_title(f'{ckpt_name} — NIH vs CheXpert')
        ax.legend()
        ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)

        # 갭 차트
        ax2 = axes[1]
        gaps = [chex_vals[i] - nih_vals[i] for i in range(len(EVAL_LABELS))]
        colors = ['red' if g < 0 else 'green' for g in gaps]
        ax2.bar(EVAL_LABELS, gaps, color=colors)
        ax2.axhline(0, color='black', linewidth=0.8)
        ax2.set_xticks(range(len(EVAL_LABELS)))
        ax2.set_xticklabels(EVAL_LABELS, rotation=40, ha='right', fontsize=9)
        ax2.set_ylabel('AUROC Gap (CheXpert − NIH)')
        ax2.set_title('Domain Shift Gap')

        plt.tight_layout()
        
        # 9. 차트 PNG 이미지 저장 (각 가중치 파일명으로 개별 저장)
        save_png_path = CHECKPOINT_DIR / f'{ckpt_name}_domain_shift.png'
        plt.savefig(str(save_png_path), dpi=150)
        plt.close(fig) # 다음 반복을 위해 메모리에서 그림 닫기
        
        print(f"✅ {ckpt_name} 산출물 저장 완료: {save_png_path}\n")

print("\n모든 모델에 대한 검증 및 저장이 완료")


[densenet_best] 도메인 시프트 검증 시작


c:\Python\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


densenet_best 추론 중:   0%|          | 0/4 [00:00<?, ?it/s]


[ densenet_best 분석 결과 ]
      Disease  NIH AUROC  CheXpert AUROC    Gap
  Atelectasis     0.8256          0.7438  -8.2%
 Cardiomegaly     0.9242          0.6680 -25.6%
Consolidation     0.8268          0.7717  -5.5%
        Edema     0.9236          0.7472 -17.6%
     Effusion     0.8962          0.7725 -12.4%
    Pneumonia     0.7714          0.6521 -11.9%
 Pneumothorax     0.8993          0.5516 -34.8%
    macro_avg     0.8667          0.7010 -16.6%
✅ densenet_best 산출물 저장 완료: checkpoints\densenet\densenet_best_domain_shift.png


[densenet_fold1_best] 도메인 시프트 검증 시작


c:\Python\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

vit_fold4_best 추론 중:   0%|          | 0/4 [00:00<?, ?it/s]


[ vit_fold4_best 분석 결과 ]
      Disease  NIH AUROC  CheXpert AUROC    Gap
  Atelectasis     0.8256          0.7895  -3.6%
 Cardiomegaly     0.9242          0.7658 -15.8%
Consolidation     0.8268          0.8307  +0.4%
        Edema     0.9236          0.7682 -15.5%
     Effusion     0.8962          0.8610  -3.5%
    Pneumonia     0.7714          0.6437 -12.8%
 Pneumothorax     0.8993          0.6315 -26.8%
    macro_avg     0.8667          0.7558 -11.1%
✅ vit_fold4_best 산출물 저장 완료: checkpoints\vit\vit_fold4_best_domain_shift.png


[vit_fold5_best] 도메인 시프트 검증 시작


c:\Python\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


vit_fold5_best 추론 중:   0%|          | 0/4 [00:00<?, ?it/s]


[ vit_fold5_best 분석 결과 ]
      Disease  NIH AUROC  CheXpert AUROC    Gap
  Atelectasis     0.8256          0.7420  -8.4%
 Cardiomegaly     0.9242          0.8261  -9.8%
Consolidation     0.8268          0.8597  +3.3%
        Edema     0.9236          0.7969 -12.7%
     Effusion     0.8962          0.8425  -5.4%
    Pneumonia     0.7714          0.5915 -18.0%
 Pneumothorax     0.8993          0.7136 -18.6%
    macro_avg     0.8667          0.7675  -9.9%
✅ vit_fold5_best 산출물 저장 완료: checkpoints\vit\vit_fold5_best_domain_shift.png


모든 모델에 대한 검증 및 저장이 완료
